# Ola Voice Assistant - Phase 1 notebook
Login / Sign up + SQLite database + voice & chat page + Groq, built with Django REST Framework.

**How to use (run top to bottom):**
1. Install packages
2. Create the project files (one cell per file)
3. Add your Groq key
4. Migrate the database, then start the server
5. Open http://127.0.0.1:8000/ in Chrome or Edge

Run this in **local Jupyter on your PC** (not Colab): the browser microphone only works on `localhost` or HTTPS.
Windows beginners: install Python 3.11/3.12, then `pip install notebook` and run `jupyter notebook`.

## 1. Install packages

In [ ]:
%pip install "Django>=5.1,<6.0" "djangorestframework>=3.15" python-dotenv requests whitenoise

## 2. Create project folder

In [ ]:
import os
PROJECT = "ola_project"
os.makedirs(PROJECT, exist_ok=True)
os.chdir(PROJECT)
for d in ["config","accounts/migrations","assistant/migrations","templates/accounts","templates/assistant","static/css","static/js","static/icons"]:
    os.makedirs(d, exist_ok=True)
print("Working in:", os.getcwd())

## 3. Write the project files
Each cell below creates one file with `%%writefile`.

In [ ]:
%%writefile config/__init__.py


In [ ]:
%%writefile config/asgi.py
import os
from django.core.asgi import get_asgi_application

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
application = get_asgi_application()


In [ ]:
%%writefile config/settings.py
import os
from pathlib import Path

from dotenv import load_dotenv

BASE_DIR = Path(__file__).resolve().parent.parent
load_dotenv(BASE_DIR / ".env")

SECRET_KEY = os.getenv("DJANGO_SECRET_KEY", "dev-only-insecure-key")
DEBUG = os.getenv("DJANGO_DEBUG", "True") == "True"
ALLOWED_HOSTS = [h.strip() for h in os.getenv("DJANGO_ALLOWED_HOSTS", "127.0.0.1,localhost").split(",") if h.strip()]
CSRF_TRUSTED_ORIGINS = [o.strip() for o in os.getenv("CSRF_TRUSTED_ORIGINS", "").split(",") if o.strip()]

INSTALLED_APPS = [
    "django.contrib.admin",
    "django.contrib.auth",
    "django.contrib.contenttypes",
    "django.contrib.sessions",
    "django.contrib.messages",
    "django.contrib.staticfiles",
    "rest_framework",
    "accounts",
    "assistant",
]

MIDDLEWARE = [
    "django.middleware.security.SecurityMiddleware",
    "whitenoise.middleware.WhiteNoiseMiddleware",
    "django.contrib.sessions.middleware.SessionMiddleware",
    "django.middleware.common.CommonMiddleware",
    "django.middleware.csrf.CsrfViewMiddleware",
    "django.contrib.auth.middleware.AuthenticationMiddleware",
    "django.contrib.messages.middleware.MessageMiddleware",
    "django.middleware.clickjacking.XFrameOptionsMiddleware",
]

ROOT_URLCONF = "config.urls"

TEMPLATES = [
    {
        "BACKEND": "django.template.backends.django.DjangoTemplates",
        "DIRS": [BASE_DIR / "templates"],
        "APP_DIRS": True,
        "OPTIONS": {
            "context_processors": [
                "django.template.context_processors.request",
                "django.contrib.auth.context_processors.auth",
                "django.contrib.messages.context_processors.messages",
            ],
        },
    },
]

WSGI_APPLICATION = "config.wsgi.application"

# SQLite: stores users, profiles, chat history. (Vector DB for RAG comes later.)
DATABASES = {
    "default": {
        "ENGINE": "django.db.backends.sqlite3",
        "NAME": BASE_DIR / "db.sqlite3",
    }
}

AUTH_PASSWORD_VALIDATORS = [
    {"NAME": "django.contrib.auth.password_validation.UserAttributeSimilarityValidator"},
    {"NAME": "django.contrib.auth.password_validation.MinimumLengthValidator"},
    {"NAME": "django.contrib.auth.password_validation.CommonPasswordValidator"},
    {"NAME": "django.contrib.auth.password_validation.NumericPasswordValidator"},
]

LANGUAGE_CODE = "en-us"
TIME_ZONE = "Asia/Kolkata"
USE_I18N = True
USE_TZ = True

STATIC_URL = "static/"
STATICFILES_DIRS = [BASE_DIR / "static"]
STATIC_ROOT = BASE_DIR / "staticfiles"
STORAGES = {
    "default": {"BACKEND": "django.core.files.storage.FileSystemStorage"},
    "staticfiles": {"BACKEND": "whitenoise.storage.CompressedStaticFilesStorage"},
}

DEFAULT_AUTO_FIELD = "django.db.models.BigAutoField"

LOGIN_URL = "/"
LOGIN_REDIRECT_URL = "/assistant/"
LOGOUT_REDIRECT_URL = "/"

REST_FRAMEWORK = {
    "DEFAULT_AUTHENTICATION_CLASSES": ["rest_framework.authentication.SessionAuthentication"],
    "DEFAULT_PERMISSION_CLASSES": ["rest_framework.permissions.IsAuthenticated"],
    "DEFAULT_THROTTLE_CLASSES": ["rest_framework.throttling.UserRateThrottle"],
    "DEFAULT_THROTTLE_RATES": {"user": "60/min"},
}

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

if not DEBUG:
    SECURE_PROXY_SSL_HEADER = ("HTTP_X_FORWARDED_PROTO", "https")
    SESSION_COOKIE_SECURE = True
    CSRF_COOKIE_SECURE = True
    SECURE_SSL_REDIRECT = os.getenv("SECURE_SSL_REDIRECT", "True") == "True"


In [ ]:
%%writefile config/urls.py
from django.contrib import admin
from django.urls import include, path

urlpatterns = [
    path("admin/", admin.site.urls),
    path("", include("accounts.urls")),
    path("", include("assistant.urls")),
]


In [ ]:
%%writefile config/wsgi.py
import os
from django.core.wsgi import get_wsgi_application

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
application = get_wsgi_application()


In [ ]:
%%writefile accounts/__init__.py


In [ ]:
%%writefile accounts/admin.py
from django.contrib import admin
from .models import UserProfile

admin.site.register(UserProfile)


In [ ]:
%%writefile accounts/apps.py
from django.apps import AppConfig


class AccountsConfig(AppConfig):
    default_auto_field = "django.db.models.BigAutoField"
    name = "accounts"


In [ ]:
%%writefile accounts/forms.py
from django import forms
from django.contrib.auth import get_user_model
from django.contrib.auth.password_validation import validate_password

from .models import UserProfile

User = get_user_model()


class RegisterForm(forms.Form):
    full_name = forms.CharField(max_length=150)
    email = forms.EmailField()
    phone = forms.CharField(max_length=20, required=False)
    password = forms.CharField(widget=forms.PasswordInput)
    confirm_password = forms.CharField(widget=forms.PasswordInput)

    def clean_email(self):
        email = self.cleaned_data["email"].lower().strip()
        if User.objects.filter(username=email).exists():
            raise forms.ValidationError("This email is already registered. Please sign in instead.")
        return email

    def clean(self):
        data = super().clean()
        pw, pw2 = data.get("password"), data.get("confirm_password")
        if pw and pw2 and pw != pw2:
            self.add_error("confirm_password", "Passwords do not match.")
        elif pw:
            try:
                validate_password(pw)
            except forms.ValidationError as e:
                self.add_error("password", e)
        return data

    def save(self):
        d = self.cleaned_data
        user = User.objects.create_user(
            username=d["email"], email=d["email"], password=d["password"],
            first_name=d["full_name"].split(" ")[0],
        )
        UserProfile.objects.create(user=user, full_name=d["full_name"], phone=d.get("phone", ""))
        return user


class LoginForm(forms.Form):
    email = forms.EmailField()
    password = forms.CharField(widget=forms.PasswordInput)


In [ ]:
%%writefile accounts/migrations/__init__.py


In [ ]:
%%writefile accounts/models.py
from django.conf import settings
from django.db import models


class UserProfile(models.Model):
    """Extra info stored for every registered user."""
    user = models.OneToOneField(settings.AUTH_USER_MODEL, on_delete=models.CASCADE, related_name="profile")
    full_name = models.CharField(max_length=150)
    phone = models.CharField(max_length=20, blank=True)
    preferred_language = models.CharField(max_length=10, default="en-IN")
    created_at = models.DateTimeField(auto_now_add=True)

    def __str__(self):
        return f"{self.full_name} <{self.user.email}>"


In [ ]:
%%writefile accounts/urls.py
from django.urls import path
from . import views

urlpatterns = [
    path("", views.auth_page, name="auth"),
    path("login/", views.login_view, name="login"),
    path("register/", views.register_view, name="register"),
    path("logout/", views.logout_view, name="logout"),
]


In [ ]:
%%writefile accounts/views.py
from django.contrib.auth import authenticate, login, logout
from django.shortcuts import redirect, render
from django.views.decorators.http import require_POST

from .forms import LoginForm, RegisterForm


def _render(request, login_form=None, register_form=None, tab="login"):
    return render(request, "accounts/auth.html", {
        "login_form": login_form or LoginForm(),
        "register_form": register_form or RegisterForm(),
        "tab": tab,
    })


def auth_page(request):
    if request.user.is_authenticated:
        return redirect("assistant")
    return _render(request, tab=request.GET.get("tab", "login"))


def login_view(request):
    if request.method != "POST":
        return redirect("auth")
    form = LoginForm(request.POST)
    if form.is_valid():
        user = authenticate(request, username=form.cleaned_data["email"].lower().strip(),
                            password=form.cleaned_data["password"])
        if user:
            login(request, user)
            return redirect("assistant")
        form.add_error(None, "Wrong email or password. New here? Use the Sign up tab.")
    return _render(request, login_form=form, tab="login")


def register_view(request):
    if request.method != "POST":
        return redirect("auth")
    form = RegisterForm(request.POST)
    if form.is_valid():
        user = form.save()
        login(request, user)
        return redirect("assistant")
    return _render(request, register_form=form, tab="register")


@require_POST
def logout_view(request):
    logout(request)
    return redirect("auth")


In [ ]:
%%writefile assistant/__init__.py


In [ ]:
%%writefile assistant/admin.py
from django.contrib import admin
from .models import Conversation, Message

admin.site.register(Conversation)
admin.site.register(Message)


In [ ]:
%%writefile assistant/api.py
from rest_framework import status
from rest_framework.response import Response
from rest_framework.views import APIView

from .groq_client import GroqError, ask_groq
from .models import Conversation, Message
from .serializers import ChatRequestSerializer, MessageSerializer

MEMORY_MESSAGES = 12  # how many past messages are sent to the LLM (conversation memory)


class ChatView(APIView):
    """POST /api/chat/  {message, conversation_id?}  ->  {reply, conversation_id}"""

    def post(self, request):
        ser = ChatRequestSerializer(data=request.data)
        ser.is_valid(raise_exception=True)
        text = ser.validated_data["message"]
        conv_id = ser.validated_data.get("conversation_id")

        conv = None
        if conv_id:
            conv = Conversation.objects.filter(id=conv_id, user=request.user).first()
        if conv is None:
            conv = Conversation.objects.create(user=request.user, title=text[:60])

        Message.objects.create(conversation=conv, role="user", content=text)
        recent = list(conv.messages.order_by("-created_at", "-id")[:MEMORY_MESSAGES])[::-1]
        history = [{"role": m.role, "content": m.content} for m in recent]

        try:
            reply = ask_groq(history)
        except GroqError as e:
            return Response({"error": str(e)}, status=status.HTTP_502_BAD_GATEWAY)

        Message.objects.create(conversation=conv, role="assistant", content=reply)
        conv.save()  # bumps updated_at
        return Response({"reply": reply, "conversation_id": conv.id})


class HistoryView(APIView):
    """GET /api/history/  -> latest conversation with its messages."""

    def get(self, request):
        conv = Conversation.objects.filter(user=request.user).first()
        if not conv:
            return Response({"conversation_id": None, "messages": []})
        return Response({
            "conversation_id": conv.id,
            "messages": MessageSerializer(conv.messages.all(), many=True).data,
        })


class NewChatView(APIView):
    """POST /api/new/  -> nothing is stored until the first message is sent."""

    def post(self, request):
        return Response({"conversation_id": None})


In [ ]:
%%writefile assistant/apps.py
from django.apps import AppConfig


class AssistantConfig(AppConfig):
    default_auto_field = "django.db.models.BigAutoField"
    name = "assistant"


In [ ]:
%%writefile assistant/groq_client.py
"""Thin wrapper around Groq's OpenAI-compatible chat completions endpoint."""
import requests
from django.conf import settings

SYSTEM_PROMPT = (
    "You are Ola, a friendly voice assistant. Your answers are spoken aloud, so keep them "
    "short, natural and conversational. Use plain text only: no markdown, bullet symbols, "
    "or emojis. If you are not sure about something, say so."
)


class GroqError(Exception):
    pass


def ask_groq(history):
    """history: list of {"role": "user"|"assistant", "content": str} (oldest first)."""
    if not settings.GROQ_API_KEY:
        raise GroqError("GROQ_API_KEY is missing. Add it to your .env file and restart the server.")

    payload = {
        "model": settings.GROQ_MODEL,
        "messages": [{"role": "system", "content": SYSTEM_PROMPT}, *history],
        "temperature": 0.6,
        "max_tokens": 500,
    }
    try:
        r = requests.post(
            f"{settings.GROQ_BASE_URL}/chat/completions",
            headers={"Authorization": f"Bearer {settings.GROQ_API_KEY}"},
            json=payload,
            timeout=30,
        )
    except requests.RequestException as e:
        raise GroqError(f"Could not reach Groq: {e}") from e

    if r.status_code != 200:
        try:
            detail = r.json().get("error", {}).get("message", r.text)
        except ValueError:
            detail = r.text
        raise GroqError(f"Groq error {r.status_code}: {detail}")

    return r.json()["choices"][0]["message"]["content"].strip()


In [ ]:
%%writefile assistant/migrations/__init__.py


In [ ]:
%%writefile assistant/models.py
from django.conf import settings
from django.db import models


class Conversation(models.Model):
    user = models.ForeignKey(settings.AUTH_USER_MODEL, on_delete=models.CASCADE, related_name="conversations")
    title = models.CharField(max_length=120, default="New chat")
    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)

    class Meta:
        ordering = ["-updated_at"]

    def __str__(self):
        return f"{self.user} - {self.title}"


class Message(models.Model):
    ROLES = [("user", "User"), ("assistant", "Assistant")]
    conversation = models.ForeignKey(Conversation, on_delete=models.CASCADE, related_name="messages")
    role = models.CharField(max_length=10, choices=ROLES)
    content = models.TextField()
    created_at = models.DateTimeField(auto_now_add=True)

    class Meta:
        ordering = ["created_at", "id"]


In [ ]:
%%writefile assistant/serializers.py
from rest_framework import serializers

from .models import Message


class ChatRequestSerializer(serializers.Serializer):
    message = serializers.CharField(max_length=2000, trim_whitespace=True)
    conversation_id = serializers.IntegerField(required=False, allow_null=True)


class MessageSerializer(serializers.ModelSerializer):
    class Meta:
        model = Message
        fields = ["role", "content", "created_at"]


In [ ]:
%%writefile assistant/urls.py
from django.urls import path
from . import api, views

urlpatterns = [
    path("assistant/", views.assistant_page, name="assistant"),
    path("sw.js", views.service_worker, name="sw"),
    path("api/chat/", api.ChatView.as_view()),
    path("api/history/", api.HistoryView.as_view()),
    path("api/new/", api.NewChatView.as_view()),
]


In [ ]:
%%writefile assistant/views.py
from django.contrib.auth.decorators import login_required
from django.http import HttpResponse
from django.shortcuts import render
from django.template.loader import render_to_string


@login_required
def assistant_page(request):
    return render(request, "assistant/chat.html")


def service_worker(request):
    """Served from the site root so it can control the whole app (PWA)."""
    js = render_to_string("assistant/sw.js")
    return HttpResponse(js, content_type="application/javascript")


In [ ]:
%%writefile templates/accounts/auth.html
{% extends "base.html" %}
{% block title %}Sign in - Ola{% endblock %}
{% block content %}
<main class="auth-wrap">
  <div class="auth-card">
    <div class="logo">&#127908;</div>
    <h1>Ola</h1>
    <p class="sub">Your voice assistant</p>

    <div class="tabs">
      <button type="button" class="tab {% if tab == 'login' %}active{% endif %}" data-tab="login">Sign in</button>
      <button type="button" class="tab {% if tab == 'register' %}active{% endif %}" data-tab="register">Sign up</button>
    </div>

    <form method="post" action="{% url 'login' %}" id="login" class="panel {% if tab == 'login' %}show{% endif %}">
      {% csrf_token %}
      {% for e in login_form.non_field_errors %}<div class="err">{{ e }}</div>{% endfor %}
      <label>Email <input type="email" name="email" required autocomplete="email" value="{{ login_form.email.value|default:'' }}"></label>
      {% for e in login_form.email.errors %}<div class="err">{{ e }}</div>{% endfor %}
      <label>Password <input type="password" name="password" required autocomplete="current-password"></label>
      <button class="btn" type="submit">Sign in</button>
      <p class="hint">Already registered? Just sign in. First time? <a href="#" data-tab="register">Create an account</a>.</p>
    </form>

    <form method="post" action="{% url 'register' %}" id="register" class="panel {% if tab == 'register' %}show{% endif %}">
      {% csrf_token %}
      {% for e in register_form.non_field_errors %}<div class="err">{{ e }}</div>{% endfor %}
      <label>Full name <input type="text" name="full_name" required value="{{ register_form.full_name.value|default:'' }}"></label>
      {% for e in register_form.full_name.errors %}<div class="err">{{ e }}</div>{% endfor %}
      <label>Email <input type="email" name="email" required value="{{ register_form.email.value|default:'' }}"></label>
      {% for e in register_form.email.errors %}<div class="err">{{ e }}</div>{% endfor %}
      <label>Phone (optional) <input type="tel" name="phone" value="{{ register_form.phone.value|default:'' }}"></label>
      <label>Password <input type="password" name="password" required autocomplete="new-password"></label>
      {% for e in register_form.password.errors %}<div class="err">{{ e }}</div>{% endfor %}
      <label>Confirm password <input type="password" name="confirm_password" required autocomplete="new-password"></label>
      {% for e in register_form.confirm_password.errors %}<div class="err">{{ e }}</div>{% endfor %}
      <button class="btn" type="submit">Create account</button>
      <p class="hint">Already have an account? <a href="#" data-tab="login">Sign in</a>.</p>
    </form>
  </div>
</main>
{% endblock %}
{% block scripts %}
<script>
  function showTab(name) {
    document.querySelectorAll(".tab").forEach(t => t.classList.toggle("active", t.dataset.tab === name));
    document.querySelectorAll(".panel").forEach(p => p.classList.toggle("show", p.id === name));
  }
  document.querySelectorAll("[data-tab]").forEach(el =>
    el.addEventListener("click", e => { e.preventDefault(); showTab(el.dataset.tab); }));
</script>
{% endblock %}


In [ ]:
%%writefile templates/assistant/chat.html
{% extends "base.html" %}
{% load static %}
{% block title %}Ola Assistant{% endblock %}
{% block content %}
<div class="app">
  <header class="topbar">
    <div class="who">Hi, <strong>{{ request.user.profile.full_name|default:request.user.first_name }}</strong></div>
    <div class="controls">
      <select id="lang" title="Speech language">
        <option value="en-IN">English (India)</option>
        <option value="hi-IN">हिन्दी</option>
        <option value="en-US">English (US)</option>
      </select>
      <button id="muteBtn" class="icon-btn" title="Toggle spoken replies">&#128266;</button>
      <button id="newBtn" class="icon-btn" title="New chat">&#10133;</button>
      <form method="post" action="{% url 'logout' %}">{% csrf_token %}<button class="icon-btn" title="Log out">&#10162;</button></form>
    </div>
  </header>

  <section id="chat" class="chat" aria-live="polite"></section>

  <footer class="dock">
    <div id="status" class="status">Tap the mic and speak, or type below</div>
    <div class="wave" id="wave"><span></span><span></span><span></span><span></span><span></span><span></span><span></span></div>
    <div class="row">
      <input id="textInput" type="text" placeholder="Type a message..." maxlength="2000" autocomplete="off">
      <button id="sendBtn" class="send">Send</button>
    </div>
    <button id="micBtn" class="mic" aria-label="Start listening">&#127908;</button>
  </footer>
</div>
{% endblock %}
{% block scripts %}
<script>window.OLA = { csrf: "{{ csrf_token }}" };</script>
<script src="{% static 'js/assistant.js' %}"></script>
{% endblock %}


In [ ]:
%%writefile templates/assistant/sw.js
// Minimal service worker: makes the app installable and caches static files.
// API calls and pages are always fetched from the network (login-protected).
const CACHE = "ola-v1";
const ASSETS = ["/static/css/style.css", "/static/js/assistant.js", "/static/manifest.json"];

self.addEventListener("install", (e) => {
  e.waitUntil(caches.open(CACHE).then((c) => c.addAll(ASSETS)));
  self.skipWaiting();
});

self.addEventListener("activate", (e) => {
  e.waitUntil(caches.keys().then((keys) => Promise.all(keys.filter((k) => k !== CACHE).map((k) => caches.delete(k)))));
});

self.addEventListener("fetch", (e) => {
  const url = new URL(e.request.url);
  if (e.request.method === "GET" && url.pathname.startsWith("/static/")) {
    e.respondWith(caches.match(e.request).then((hit) => hit || fetch(e.request)));
  }
});


In [ ]:
%%writefile templates/base.html
{% load static %}
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
  <title>{% block title %}Ola Voice Assistant{% endblock %}</title>
  <meta name="theme-color" content="#6c5ce7">
  <link rel="manifest" href="{% static 'manifest.json' %}">
  <link rel="icon" href="{% static 'icons/icon-192.png' %}">
  <link rel="apple-touch-icon" href="{% static 'icons/icon-192.png' %}">
  <link rel="stylesheet" href="{% static 'css/style.css' %}">
</head>
<body>
  {% block content %}{% endblock %}
  <script>
    if ("serviceWorker" in navigator) {
      navigator.serviceWorker.register("/sw.js").catch(() => {});
    }
  </script>
  {% block scripts %}{% endblock %}
</body>
</html>


In [ ]:
%%writefile static/css/style.css
:root { --bg:#0f1020; --card:#1a1b33; --acc:#6c5ce7; --acc2:#a29bfe; --txt:#eef; --mut:#9aa; --err:#ff7675; }
* { box-sizing: border-box; }
html, body { height: 100%; margin: 0; }
body { background: var(--bg); color: var(--txt); font-family: system-ui, "Segoe UI", sans-serif; }
a { color: var(--acc2); }

/* ---------- auth ---------- */
.auth-wrap { min-height: 100%; display: grid; place-items: center; padding: 20px; }
.auth-card { width: 100%; max-width: 400px; background: var(--card); padding: 28px; border-radius: 20px; box-shadow: 0 10px 40px #0008; text-align: center; }
.logo { font-size: 42px; } h1 { margin: 4px 0 0; } .sub { color: var(--mut); margin: 0 0 18px; }
.tabs { display: flex; background: #0003; border-radius: 12px; padding: 4px; margin-bottom: 16px; }
.tab { flex: 1; padding: 10px; background: none; border: 0; color: var(--mut); border-radius: 9px; cursor: pointer; font-size: 15px; }
.tab.active { background: var(--acc); color: #fff; }
.panel { display: none; text-align: left; } .panel.show { display: block; }
label { display: block; font-size: 13px; color: var(--mut); margin-top: 12px; }
input, select { width: 100%; margin-top: 4px; padding: 12px; border-radius: 10px; border: 1px solid #334; background: #0f1020; color: var(--txt); font-size: 16px; }
.btn { width: 100%; margin-top: 18px; padding: 13px; border: 0; border-radius: 12px; background: var(--acc); color: #fff; font-size: 16px; cursor: pointer; }
.err { color: var(--err); font-size: 13px; margin-top: 4px; } .hint { font-size: 13px; color: var(--mut); text-align: center; }

/* ---------- assistant ---------- */
.app { height: 100%; max-width: 760px; margin: 0 auto; display: flex; flex-direction: column; }
.topbar { display: flex; justify-content: space-between; align-items: center; padding: 12px 16px; padding-top: max(12px, env(safe-area-inset-top)); }
.controls { display: flex; gap: 8px; align-items: center; } .controls select { width: auto; margin: 0; padding: 8px; font-size: 13px; }
.controls form { margin: 0; }
.icon-btn { background: var(--card); border: 0; color: var(--txt); border-radius: 10px; padding: 9px 11px; cursor: pointer; font-size: 15px; }
.chat { flex: 1; overflow-y: auto; padding: 8px 16px; display: flex; flex-direction: column; gap: 10px; }
.msg { max-width: 82%; padding: 11px 14px; border-radius: 16px; line-height: 1.45; white-space: pre-wrap; word-wrap: break-word; }
.msg.user { align-self: flex-end; background: var(--acc); border-bottom-right-radius: 4px; }
.msg.assistant { align-self: flex-start; background: var(--card); border-bottom-left-radius: 4px; }
.msg.error { align-self: center; background: #ff767522; color: var(--err); font-size: 14px; }
.msg.typing { color: var(--mut); font-style: italic; }
.dock { padding: 10px 16px calc(14px + env(safe-area-inset-bottom)); text-align: center; position: relative; }
.status { color: var(--mut); font-size: 13px; min-height: 18px; }
.row { display: flex; gap: 8px; margin-top: 8px; } .row input { margin: 0; }
.send { background: var(--card); color: var(--txt); border: 0; border-radius: 10px; padding: 0 18px; cursor: pointer; }
.mic { margin: 14px auto 0; display: block; width: 84px; height: 84px; border-radius: 50%; border: 0; font-size: 34px; color: #fff; cursor: pointer;
       background: radial-gradient(circle at 30% 30%, var(--acc2), var(--acc)); box-shadow: 0 0 0 0 #6c5ce788; transition: transform .15s; }
.mic:active { transform: scale(.94); }
.mic.listening { animation: pulse 1.4s infinite; background: radial-gradient(circle at 30% 30%, #ff7675, #d63031); }
@keyframes pulse { 0% { box-shadow: 0 0 0 0 #ff767588; } 70% { box-shadow: 0 0 0 26px #ff767500; } 100% { box-shadow: 0 0 0 0 #ff767500; } }
.wave { display: none; height: 34px; justify-content: center; align-items: center; gap: 4px; margin-top: 6px; }
.wave.on { display: flex; }
.wave span { width: 5px; height: 8px; border-radius: 3px; background: var(--acc2); animation: bar 0.9s ease-in-out infinite; }
.wave span:nth-child(2) { animation-delay: .1s } .wave span:nth-child(3) { animation-delay: .2s } .wave span:nth-child(4) { animation-delay: .3s }
.wave span:nth-child(5) { animation-delay: .2s } .wave span:nth-child(6) { animation-delay: .1s }
@keyframes bar { 0%, 100% { height: 8px; } 50% { height: 32px; } }


In [ ]:
%%writefile static/js/assistant.js
(() => {
  const $ = (id) => document.getElementById(id);
  const chat = $("chat"), micBtn = $("micBtn"), wave = $("wave"), statusEl = $("status");
  const input = $("textInput"), sendBtn = $("sendBtn"), langSel = $("lang"), muteBtn = $("muteBtn");

  let conversationId = null;
  let muted = localStorage.getItem("ola_muted") === "1";
  let busy = false;

  langSel.value = localStorage.getItem("ola_lang") || "en-IN";
  langSel.onchange = () => localStorage.setItem("ola_lang", langSel.value);
  const paintMute = () => (muteBtn.textContent = muted ? "\u{1F507}" : "\u{1F50A}");
  paintMute();
  muteBtn.onclick = () => { muted = !muted; localStorage.setItem("ola_muted", muted ? "1" : "0"); paintMute(); if (muted) speechSynthesis.cancel(); };

  // ---------- chat UI ----------
  function addMsg(role, text, cls = "") {
    const d = document.createElement("div");
    d.className = `msg ${role} ${cls}`.trim();
    d.textContent = text; // textContent => safe against HTML injection
    chat.appendChild(d);
    chat.scrollTop = chat.scrollHeight;
    return d;
  }

  async function loadHistory() {
    try {
      const r = await fetch("/api/history/");
      const data = await r.json();
      conversationId = data.conversation_id;
      data.messages.forEach((m) => addMsg(m.role, m.content));
      if (!data.messages.length) addMsg("assistant", "Hi! I'm Ola. Tap the mic and ask me anything.");
    } catch { addMsg("error", "Could not load history.", "error"); }
  }

  // ---------- talking to the Django API ----------
  async function send(text) {
    text = text.trim();
    if (!text || busy) return;
    busy = true;
    speechSynthesis.cancel();
    addMsg("user", text);
    input.value = "";
    const typing = addMsg("assistant", "Thinking...", "typing");
    statusEl.textContent = "Thinking...";
    try {
      const r = await fetch("/api/chat/", {
        method: "POST",
        headers: { "Content-Type": "application/json", "X-CSRFToken": window.OLA.csrf },
        body: JSON.stringify({ message: text, conversation_id: conversationId }),
      });
      const data = await r.json();
      typing.remove();
      if (!r.ok) {
        addMsg("error", data.error || data.detail || "Something went wrong.", "error");
      } else {
        conversationId = data.conversation_id;
        addMsg("assistant", data.reply);
        speak(data.reply);
      }
    } catch {
      typing.remove();
      addMsg("error", "Network error. Please try again.", "error");
    } finally {
      busy = false;
      statusEl.textContent = "Tap the mic and speak, or type below";
    }
  }

  sendBtn.onclick = () => send(input.value);
  input.addEventListener("keydown", (e) => { if (e.key === "Enter") send(input.value); });
  $("newBtn").onclick = () => {
    conversationId = null; chat.innerHTML = ""; speechSynthesis.cancel();
    addMsg("assistant", "New chat started. How can I help?");
  };

  // ---------- text-to-speech (browser) ----------
  function speak(text) {
    if (muted || !("speechSynthesis" in window)) return;
    const u = new SpeechSynthesisUtterance(text);
    u.lang = langSel.value;
    const voice = speechSynthesis.getVoices().find((v) => v.lang === langSel.value);
    if (voice) u.voice = voice;
    speechSynthesis.speak(u);
  }

  // ---------- speech-to-text (browser) ----------
  const SR = window.SpeechRecognition || window.webkitSpeechRecognition;
  let rec = null, listening = false;

  if (!SR) {
    micBtn.disabled = true;
    statusEl.textContent = "Voice input isn't supported in this browser. Use Chrome or Edge, or type.";
  } else {
    rec = new SR();
    rec.interimResults = true;
    rec.continuous = false;

    rec.onstart = () => { listening = true; micBtn.classList.add("listening"); wave.classList.add("on"); statusEl.textContent = "Listening..."; };
    rec.onend = () => { listening = false; micBtn.classList.remove("listening"); wave.classList.remove("on"); };
    rec.onerror = (e) => {
      const msg = { "not-allowed": "Microphone blocked. Allow mic access in the browser.", "no-speech": "I didn't hear anything.", "network": "Speech service unavailable (needs internet)." };
      statusEl.textContent = msg[e.error] || `Mic error: ${e.error}`;
    };
    rec.onresult = (e) => {
      let finalText = "", interim = "";
      for (const res of e.results) (res.isFinal ? (finalText += res[0].transcript) : (interim += res[0].transcript));
      input.value = finalText || interim;
      if (finalText) send(finalText);
    };

    micBtn.onclick = () => {
      if (listening) { rec.stop(); return; }
      speechSynthesis.cancel();
      rec.lang = langSel.value;
      try { rec.start(); } catch {}
    };
  }

  loadHistory();
})();


In [ ]:
%%writefile static/manifest.json
{
  "name": "Ola Voice Assistant",
  "short_name": "Ola",
  "start_url": "/assistant/",
  "scope": "/",
  "display": "standalone",
  "background_color": "#0f1020",
  "theme_color": "#6c5ce7",
  "icons": [
    { "src": "/static/icons/icon-192.png", "sizes": "192x192", "type": "image/png" },
    { "src": "/static/icons/icon-512.png", "sizes": "512x512", "type": "image/png", "purpose": "any maskable" }
  ]
}


In [ ]:
%%writefile .env.example
# Copy this file to .env and fill in the values
DJANGO_SECRET_KEY=change-me-to-a-long-random-string
DJANGO_DEBUG=True
DJANGO_ALLOWED_HOSTS=127.0.0.1,localhost
# Add your domain after deploying, e.g. https://ola.yourdomain.com
CSRF_TRUSTED_ORIGINS=

# Get a free key at https://console.groq.com/keys
GROQ_API_KEY=
# Model IDs change over time - check https://console.groq.com/docs/models
GROQ_MODEL=llama-3.3-70b-versatile


In [ ]:
%%writefile .gitignore
venv/
.env
db.sqlite3
__pycache__/
*.pyc
staticfiles/


In [ ]:
%%writefile manage.py
#!/usr/bin/env python
import os
import sys


def main():
    os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
    from django.core.management import execute_from_command_line
    execute_from_command_line(sys.argv)


if __name__ == "__main__":
    main()


### App icons (PWA)

In [ ]:
import zlib, struct
def png(size, path, rgb=(108,92,231)):
    raw = b"".join(b"\x00" + bytes(rgb) * size for _ in range(size))
    def chunk(t, d):
        c = struct.pack(">I", len(d)) + t + d
        return c + struct.pack(">I", zlib.crc32(t + d) & 0xffffffff)
    data = b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", size, size, 8, 2, 0, 0, 0)) + chunk(b"IDAT", zlib.compress(raw)) + chunk(b"IEND", b"")
    open(path, "wb").write(data)
png(192, "static/icons/icon-192.png"); png(512, "static/icons/icon-512.png")
print("icons created")

## 4. Settings and Groq key
Get a free key at https://console.groq.com/keys. The key is saved only in the local `.env` file.

In [ ]:
from getpass import getpass
import secrets
key = getpass("Paste your Groq API key (input hidden): ")
with open(".env", "w") as f:
    f.write(f"""DJANGO_SECRET_KEY={secrets.token_urlsafe(50)}
DJANGO_DEBUG=True
DJANGO_ALLOWED_HOSTS=127.0.0.1,localhost
CSRF_TRUSTED_ORIGINS=
GROQ_API_KEY={key}
GROQ_MODEL=llama-3.3-70b-versatile
""")
print(".env written")

## 5. Create the database

In [ ]:
!python manage.py makemigrations accounts assistant
!python manage.py migrate

### (Optional) Create an admin user to view stored data at /admin/
Run this in a terminal instead: `python manage.py createsuperuser`

## 6. Start the server
Runs Django in the background. Then open http://127.0.0.1:8000/

In [ ]:
import subprocess, sys
server = subprocess.Popen([sys.executable, "manage.py", "runserver", "8000", "--noreload"])
print("Server started -> http://127.0.0.1:8000/")

## 7. Stop the server (when finished)

In [ ]:
server.terminate()
print('Server stopped')

## Next steps
- Phase 2: deploy to the cloud, point your domain's DNS (CNAME) at it, enable HTTPS (needed for phone microphone).
- Phase 3: PDF upload + RAG (pypdf, ChromaDB, embeddings).
- Phase 4: optional Whisper / KittenTTS / local Ollama.

If Groq says "model not found", change `GROQ_MODEL` in `.env` (see https://console.groq.com/docs/models) and restart the server.